# 05 — Where are Lagos's CNG stations?

**Question:** how are the known CNG stations spread across Lagos, and how much of the city is plausibly within easy
reach of one?

**Data:** `data/processed/lagos_cng_stations_geocoded.csv`, made by `src/geocode_stations.py` from
`data/raw/lagos_cng_stations.csv` (22 named sites; sources S34–S37, S39 in `docs/sources.md`).

**Positions are approximate.** 15 of the 22 sites could only be placed at the centre of their neighbourhood
(`geocode_precision = "area"`), which can be several kilometres from the actual station. The other 7 matched a
named road or place, which is closer but still not the forecourt itself.

## 1. Setup and load

We use **folium**, a Python wrapper for the Leaflet web-map library. It writes a self-contained HTML map you can pan
and zoom. Leaflet and the map tiles load from the internet when the file is opened.

In [1]:
from pathlib import Path

import folium
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

stations = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "lagos_cng_stations_geocoded.csv")
assert stations["latitude"].notna().all(), "some stations have no coordinates - re-run geocode_stations.py"
print(f"{len(stations)} stations")
print(stations["geocode_precision"].value_counts().to_string())

22 stations
geocode_precision
area        15
location     7


## 2. Status groups

`status_reported` has five spellings. We group them into three, ordered from most to least certain:

| Group | `status_reported` | Stations | Evidence |
|---|---|---|---|
| Confirmed operational | Operational | L01–L06 | 2 sources each |
| Commissioned, operation not confirmed | Commissioned, Commissioned (network) | L11–L22 | 1 source each, except L11 (Portland Gas mother station) with 2 |
| Planned / unconfirmed | Planned (end Jan 2025) | L07–L10 | Opening not confirmed |

"Commissioned" means a launch was announced. It doesn't confirm that gas is being sold there today. The 11 IBILE
(IOGC) sites all come from one source (S37), so their group is labelled by status rather than "1 source".

In [2]:
STATUS_GROUPS = {
    "Operational": "Confirmed operational",
    "Commissioned": "Commissioned, operation not confirmed",
    "Commissioned (network)": "Commissioned, operation not confirmed",
    "Planned (end Jan 2025)": "Planned / unconfirmed",
}
GROUP_ORDER = ["Confirmed operational", "Commissioned, operation not confirmed", "Planned / unconfirmed"]

stations["status_group"] = stations["status_reported"].map(STATUS_GROUPS)
assert stations["status_group"].notna().all(), \
    f"unmapped status: {stations.loc[stations['status_group'].isna(), 'status_reported'].unique()}"

# The raw operator column spells IBILE two ways; group them for counting only.
stations["operator_group"] = stations["operator"].replace(
    {"IBILE Oil & Gas Corp (Lagos State)": "IBILE Oil & Gas Corp"}
)

## 3. Counts by status and operator

In [3]:
by_status = (stations.groupby("status_group")
             .agg(stations=("station_id", "size"), sources_per_site=("sources_count", lambda s: f"{s.min()}–{s.max()}"
                                                                   if s.min() != s.max() else f"{s.min()}"))
             .reindex(GROUP_ORDER))
by_status.loc["Total"] = [len(stations), ""]
by_status

,stations,sources_per_site
status_group,,
Confirmed operational,6,2
"Commissioned, operation not confirmed",12,1–2
Planned / unconfirmed,4,2
Total,22,


In [4]:
# pd.crosstab counts stations for every operator x status combination.
by_operator = pd.crosstab(stations["operator_group"], stations["status_group"], margins=True, margins_name="Total")
by_operator = by_operator.reindex(columns=GROUP_ORDER + ["Total"], fill_value=0)
# Sort operators by size, keeping the Total row at the bottom.
operators = by_operator.drop(index="Total").sort_values("Total", ascending=False)
pd.concat([operators, by_operator.loc[["Total"]]])

status_group,Confirmed operational,"Commissioned, operation not confirmed",Planned / unconfirmed,Total
operator_group,,,,
IBILE Oil & Gas Corp,0,11,0,11
NNPC Retail,2,0,4,6
NNPC Retail / NIPCO,4,0,0,4
Portland Gas,0,1,0,1
Total,6,12,4,22


## 4. The map

Design choices:
- **Colour shows how certain we are that a station exists and works.** Dark blue for confirmed, mid blue for
  commissioned, and a hollow grey ring for planned. Because the planned marker is a different shape (hollow), the
  groups stay distinguishable without relying on colour.
- **OpenStreetMap basemap**, which needs no API key. It is busier than a plain grey map, so markers have a white
  outline to stand out.
- **3 km circles** around confirmed and commissioned stations, as a rough picture of "nearby". A circle is straight-line
  distance, not travel time; Lagos traffic can make 3 km take far longer than 10 minutes. The circles are a separate
  layer you can switch off (top-right control).
- **Stacked points are nudged apart.** Two pairs of stations were geocoded to the same neighbourhood centre (L07 and
  L11 at "Kosofe"; L06 and L15 at "Agege"). For display only, each second point in a pair is shifted about 300 m east
  so both markers can be clicked. The saved CSV is not changed.

In [5]:
COLORS = {
    "Confirmed operational": "#104281",
    "Commissioned, operation not confirmed": "#5598e7",
    "Planned / unconfirmed": "#8a8984",
}
REACH_KM = 3
REACH_LABEL = "approx. 10-minute reach (illustrative, not travel time)"

# Display offset for stations that share exact coordinates (see note above).
plot = stations.copy()
dup_rank = plot.groupby(["latitude", "longitude"]).cumcount()   # 0 for the first at a spot, 1 for the next...
plot["lon_display"] = plot["longitude"] + dup_rank * 0.0027      # 0.0027 deg of longitude ~ 300 m at Lagos
print("Nudged for display:", plot.loc[dup_rank > 0, "station_id"].tolist())

m = folium.Map(tiles=None, control_scale=True)
# OpenStreetMap tiles need no API key (CartoDB's now do). tiles=None above + TileLayer here lets us name
# the layer in the layer control.
folium.TileLayer("OpenStreetMap", name="Street map (OpenStreetMap)").add_to(m)
# Zoom to fit every station, with a little padding.
m.fit_bounds([[plot["latitude"].min(), plot["lon_display"].min()],
              [plot["latitude"].max(), plot["lon_display"].max()]], padding=(30, 30))

reach_layer = folium.FeatureGroup(name=f"{REACH_KM} km circles: {REACH_LABEL}", show=True)
for group in GROUP_ORDER:
    layer = folium.FeatureGroup(name=group, show=True)
    for _, s in plot[plot["status_group"] == group].iterrows():
        popup_html = (
            f"<b>{s['name_or_location']}</b> ({s['station_id']})<br>"
            f"Area: {s['area_lga']}<br>"
            f"Operator: {s['operator']}<br>"
            f"Status: {s['status_reported']} ({s['date_reported']})<br>"
            f"Confidence: {s['confidence']} · sources: {s['source_ids']}<br>"
            f"Geocode precision: <b>{s['geocode_precision']}</b>"
            + (" (neighbourhood centre, may be km off)" if s["geocode_precision"] == "area" else "")
        )
        planned = group == "Planned / unconfirmed"
        folium.CircleMarker(
            location=[s["latitude"], s["lon_display"]],
            radius=7,
            color="white" if not planned else COLORS[group],      # outline
            weight=2 if not planned else 3,
            fill=True,
            fill_color=COLORS[group],
            fill_opacity=0.0 if planned else 1.0,                  # hollow ring for planned
            popup=folium.Popup(popup_html, max_width=320),
            tooltip=f"{s['station_id']}: {s['name_or_location']} ({group})",
        ).add_to(layer)
        if not planned:
            folium.Circle(
                location=[s["latitude"], s["lon_display"]],
                radius=REACH_KM * 1000,            # metres, drawn to true map scale
                color=COLORS[group], weight=1, opacity=0.5,
                fill=True, fill_color=COLORS[group], fill_opacity=0.08,
                interactive=False,                 # clicks go through to the station marker
            ).add_to(reach_layer)
    layer.add_to(m)
reach_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

Nudged for display: ['L11', 'L15']


### Legend and note

folium has no built-in legend for markers, so we add a small HTML box fixed to the corner of the map. A second box
carries the note that positions are approximate.

In [6]:
n_area = (stations["geocode_precision"] == "area").sum()
swatch = lambda color, hollow=False: (
    f'<span style="display:inline-block;width:12px;height:12px;border-radius:50%;margin-right:6px;'
    f'vertical-align:-1px;{"border:3px solid " + color if hollow else "background:" + color + ";border:2px solid white"};'
    f'box-sizing:border-box"></span>'
)
legend_html = f"""
<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:white;padding:10px 12px;
            border:1px solid #d6d5cf;border-radius:6px;font:13px/1.5 Inter,Segoe UI,Arial,sans-serif;color:#0b0b0b;
            max-width:300px">
  <b>Lagos CNG stations (22 named sites)</b><br>
  {swatch(COLORS['Confirmed operational'])}Confirmed operational (2 sources)<br>
  {swatch(COLORS['Commissioned, operation not confirmed'])}Commissioned, operation not confirmed<br>
  {swatch(COLORS['Planned / unconfirmed'], hollow=True)}Planned / unconfirmed<br>
  <span style="display:inline-block;width:12px;height:12px;border-radius:50%;margin-right:6px;vertical-align:-1px;
               background:rgba(85,152,231,0.15);border:1px solid #5598e7;box-sizing:border-box"></span>{REACH_KM} km circle:
  {REACH_LABEL}
</div>
<div style="position:fixed;top:12px;left:56px;z-index:9999;background:white;padding:6px 10px;
            border:1px solid #d6d5cf;border-radius:6px;font:12px/1.4 Inter,Segoe UI,Arial,sans-serif;color:#52514e;
            max-width:360px">
  <b style="color:#0b0b0b">Positions are approximate.</b> {n_area} of {len(stations)} sites are placed at their
  neighbourhood centre (area-level geocoding), not the station itself. Sources: docs/sources.md S34–S37, S39.
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

out_path = FIG_DIR / "lagos_station_map.html"
m.save(out_path)
print(f"Saved {out_path.relative_to(PROJECT_ROOT)}")
m

Saved outputs\figures\lagos_station_map.html


## 5. What the map shows

**Only 6 refuelling points (4 fixed stations and 2 mobile units) of the 22 named sites are confirmed
operational.** All are NNPC Retail sites, with the 4 fixed stations run jointly with NIPCO, and each is reported
by two sources. A further 12 have been commissioned but are not
confirmed to be selling gas: 11 IBILE (IOGC) stations, all from a single source (S37), and the Portland Gas mother
station at Ojota. The remaining 4 are NNPC mobile units that were planned for January 2025, with no confirmation that
they opened.

**The 6 confirmed refuelling points are concentrated in a few parts of the mainland.** Five are in the central and northern
mainland (Ikeja, Mushin, Shomolu, Agege and Apapa) and one is on the Lekki–Epe corridor at Sangotedo. Large parts of
the state, including Alimosho, Ikorodu, Badagry, Ibeju-Lekki and Lagos Island, have no confirmed station on this list.
Those areas have only commissioned or planned sites, if any. Even on the illustrative 3 km circles, most of the
state lies outside reach of a confirmed station. If the commissioned IBILE network is working, coverage looks much
better, especially in Alimosho, Kosofe and the Ajah/Ikota area. Which picture is true depends on evidence we don't yet
have.

**This fits what petrol drivers told us.** In the driver survey (S47, notebook 04), 10 of the 12 petrol-only drivers
said there is no CNG station near where they work, the most common barrier. A thin, clustered network of confirmed
stations is consistent with that. The survey didn't record where drivers work, though, so we can't match individual
drivers to the map. The map shows that the complaint is plausible, not that it is true for each driver.

**Limitations:**
- **Approximate positions.** 15 of the 22 sites are placed at the centre of their neighbourhood, not at the station,
  and can be several kilometres off. Two pairs share a point and are nudged apart for display. The other 7 matched a
  road or place name, not a verified forecourt location.
- **22 named sites only.** This is the list compiled from published sources (S34–S37, S39). Stations that were never
  reported in the sources we found are missing, and some listed sites may have closed.
- **Circles are straight-line distance, not travel time.** 3 km is a rough stand-in for "about 10 minutes away". In
  Lagos traffic, reaching a station 3 km away can take much longer, and water and road layout mean some points inside
  a circle are far from the station by road.
- **The station list may be incomplete or out of date.** Most entries were reported in January 2025 or May 2026, and
  the status of the commissioned and planned sites hasn't been checked since. The NMDPRA list of licensed CNG
  stations (under "Not yet reviewed" in `docs/sources.md`) would be the best check.